# 1. Mount google drive

In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Set working directory and clone repository dynamically

import os, sys, shutil, json

# 1. Establish persistent folder on Google Drive for caching
drive_hvac_dir = "/content/drive/MyDrive/SmartBEM-Studio"
os.makedirs(drive_hvac_dir, exist_ok=True)

# 2. Establish local repository runtime folder
local_repo_path = "/content/SmartBEM-Studio"
%cd /content
if os.path.exists(local_repo_path):
    shutil.rmtree(local_repo_path)

print("Cloning repository from GitHub...")
!git clone https://github.com/kethshen/SmartBEM-Studio.git {local_repo_path}

colab_dir = os.path.join(local_repo_path, "backend_server")

# 3. Ensure eplus utility is copied for imports
eplus_src = os.path.join(local_repo_path, "EnergyPlus utility")
eplus_dest = os.path.join(colab_dir, "eplus")
if not os.path.exists(eplus_dest):
    shutil.copytree(eplus_src, eplus_dest)

%cd {colab_dir}
sys.path.append(colab_dir)
print(f"Working directory set to: {os.getcwd()}")

# 4. Load Ngrok Authtoken from Google Colab Secrets
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')

    # Save it to local secrets.json so fastapi_server.py can parse it
    local_secrets = os.path.join(colab_dir, "secrets.json")
    with open(local_secrets, "w") as f:
        json.dump({"NGROK_AUTHTOKEN": ngrok_token}, f, indent=4)
    print("✅ Successfully loaded NGROK_AUTHTOKEN from Colab Secrets!")
except Exception as e:
    print("⚠️ Colab Secret 'NGROK_AUTHTOKEN' not found or failed to load.")
    print("Please click the key icon (🔑) in the Colab sidebar, add a secret named 'NGROK_AUTHTOKEN', grant access, and rerun.")


/content
Cloning repository from GitHub...
Cloning into '/content/SmartBEM-Studio'...
remote: Enumerating objects: 1098, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 1098 (delta 56), reused 77 (delta 31), pack-reused 973 (from 1)
Receiving objects: 100% (1098/1098), 70.77 MiB | 25.61 MiB/s, done.
Resolving deltas: 100% (667/667), done.
/content/SmartBEM-Studio/backend_server
Working directory set to: /content/SmartBEM-Studio/backend_server
✅ Successfully loaded NGROK_AUTHTOKEN from Colab Secrets!


## Install openstudio


In [3]:
# Run this in a new cell at the top of your notebook:
from eplus import prepare_colab_eplus
prepare_colab_eplus()


[Bootstrap] Installing openstudio python bindings (v3.10.0)...


## Setup NGROK

In [4]:
!pip install pyngrok nest-asyncio fastapi uvicorn pydantic


In [15]:
from core.fastapi_server import start_server
start_server(port=8000)


************************************************************
🚀 SMART HVAC BACKEND IS LIVE!
🔗 COPY THIS URL TO YOUR WEB DASHBOARD: https://glaucoma-concert-outer.ngrok-free.dev
📄 VIEW SYSTEM LOGS AT: https://glaucoma-concert-outer.ngrok-free.dev/results/backend.log
💡 TO VIEW LIVE LOGS IN COLAB RUN: !tail -n 100 -f /tmp/smartbem_sim_runs/backend.log
************************************************************
Server is running in the background. You can continue using the notebook.


## 2. Ollama local run

In [5]:
# 1. Install missing dependencies & GPU utils
!sudo apt-get install -y zstd
!sudo apt update && sudo apt install pciutils lshw -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 91 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (9,041 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122464 files and directories currently

In [6]:
# 2. Install Ollama Linux Engine
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
# 3. Install Python SDK
!pip install ollama

## Install Ollama models

In [ ]:
import os
os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/SmartBEM-Studio/ollama_models/'


In [ ]:
!ollama pull gemma3:12b


In [ ]:
import subprocess
# Remove gemma manifest
!rm -rf "'/content/drive/MyDrive/SmartBEM-Studio/ollama_models/manifests/registry.ollama.ai/library/gemma3"


In [ ]:
# Easiest: just run ollama rm while OLLAMA_MODELS points to Drive
import os
os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/SmartBEM-Studio/ollama_models"
!ollama rm gemma3:4b


## Copy model

In [8]:
# 2. Safely copy the massive brain from Drive to Colab's fast internal disk (~90 seconds)
import os
import shutil

drive_models_dir = "/content/drive/MyDrive/SmartBEM-Studio/ollama_models"
local_models_dir = "/root/.ollama/models"

if os.path.exists(drive_models_dir) and os.listdir(drive_models_dir):
    print("Copying model from Drive to local disk...")
    os.makedirs(local_models_dir, exist_ok=True)
    for item in os.listdir(drive_models_dir):
        s = os.path.join(drive_models_dir, item)
        d = os.path.join(local_models_dir, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    print("Copy complete!")
else:
    print("No cached models found on Google Drive. Will download on-demand.")


Copying model from Drive to local disk...
Copy complete!


In [9]:
# 4. Set host environment variables to allow connections
import os
import time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

In [10]:
# 3. Boot Server in a directory that will never be deleted to avoid getcwd errors on rerun
# First, kill any existing stale ollama instances
!pkill -f ollama
time.sleep(1)

# Move to /content to start the background process safely
%cd /content
!nohup ollama serve > /content/ollama_serve.log 2>&1 &
time.sleep(5)

# Return to the active workspace directory
%cd /content/SmartBEM-Studio/backend_server
print("✅ Ollama Server Awake and ready!")


/content
/content/SmartBEM-Studio/backend_server
✅ Ollama Server Awake and ready!


In [11]:
# 6. Test if it is running
!curl http://localhost:11434

Ollama is running

In [12]:
!ollama list

NAME          ID              SIZE      MODIFIED       
gemma3:12b    f4031aab637d    8.1 GB    53 minutes ago    


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Test model

In [13]:
"""### Testing on Text Input"""

import ollama
response = ollama.chat(model='gemma3:12b', messages=[
  {'role': 'user', 'content': 'what LLM model are you, answer short'},
])
print(response['message']['content'])

I'm Gemma, a large language model created by Google DeepMind.


## 2. Setup Environment

In [14]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

INFO: pip is looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.7 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies installed.


In [ ]:
# Ensure the directory and log file exist, then tail
!mkdir -p /tmp/smartbem_sim_runs && touch /tmp/smartbem_sim_runs/backend.log



In [ ]:
!tail -n 100 -f /tmp/smartbem_sim_runs/backend.log
